# E5 — Imbalance-handling ablation: RFS ON vs OFF (YOLO26s, 15 epochs)
Isolates the contribution of repeat-factor sampling: same model, same seed 42, same 15 epochs —
one run WITH RFS, one WITHOUT. Compares overall mAP and the AP of the rare classes (no_vest, boots, mask, no_mask, no_boots).

**Add Input:** dataset `ppe-unified-v1`. GPU T4 x2 · Internet ON. Estimated ~3-3.5 h. Checkpoint-resume supported.

In [ ]:
# ==================== CONFIG (E5) ====================
QUICK_TEST = False
SEED = 42                      # seed for the data copy (fixed); the training seed varies per experiment
RFS_T, REPEAT_CAP = 0.15, 4

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: load the checkpoint from THIS notebook's previous output (if Added as Input)
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith("e5_") for k in _s):
        shutil.copy(_st, STATE_PATH)
        _src_runs = os.path.join(os.path.dirname(_st), "runs")
        if os.path.isdir(_src_runs): shutil.copytree(_src_runs, f"{WORK}/runs", dirs_exist_ok=True)
        print("CHECKPOINT LOADED FROM:", _st); break
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()
def free():
    import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("[E5] stages done:", list(STATE) or "(none yet)")

In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Enable GPU T4 x2!"
cap = torch.cuda.get_device_capability(0); assert cap[0] >= 7, "GPU too old - select T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

In [ ]:
from pathlib import Path
from collections import Counter
from math import sqrt
import yaml

def prepare_dataset(dst_name, use_rfs=True):
    """Copy PPE_Unified_v1 from /kaggle/input into working; use_rfs=True oversamples rare classes (train split)."""
    DS_ROOT = None
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if os.path.isdir(f"{root}/train/images") and os.path.isdir(f"{root}/val/images") and os.path.isdir(f"{root}/test/images"):
            DS_ROOT = root; break
    assert DS_ROOT, "Dataset ppe-unified-v1 has not been Added as Input!"
    raw = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    if isinstance(raw, dict): raw = [raw[k] for k in sorted(raw)]
    CLASSES = list(raw); NUM_CLS = len(CLASSES)

    DATA = f"{WORK}/{dst_name}"
    splits = {}
    for sp in ["train", "val", "test"]:
        os.makedirs(f"{DATA}/{sp}/images", exist_ok=True); os.makedirs(f"{DATA}/{sp}/labels", exist_ok=True)
        files = sorted(os.listdir(f"{DS_ROOT}/{sp}/images"))
        if len(os.listdir(f"{DATA}/{sp}/images")) < len(files):
            for fn in files:
                shutil.copy(f"{DS_ROOT}/{sp}/images/{fn}", f"{DATA}/{sp}/images/{fn}")
                lp = f"{DS_ROOT}/{sp}/labels/{Path(fn).stem}.txt"
                if os.path.exists(lp): shutil.copy(lp, f"{DATA}/{sp}/labels/{Path(fn).stem}.txt")
        splits[sp] = files

    if use_rfs:
        img_classes = {}
        for lf in os.listdir(f"{DATA}/train/labels"):
            if "_rfs" in lf: continue
            present = Counter()
            for line in open(f"{DATA}/train/labels/{lf}"):
                if line.strip(): present[int(line.split()[0])] += 1
            img_classes[lf] = present
        n_train = len(splits["train"])
        f_c = {c: sum(1 for s in img_classes.values() if c in s)/max(1, n_train) for c in range(NUM_CLS)}
        r_c = {c: max(1.0, sqrt(RFS_T/max(f_c[c], 1e-6))) for c in range(NUM_CLS)}
        added = 0
        for lf, present in img_classes.items():
            if not present: continue
            r = min(REPEAT_CAP, max(r_c[c] for c in present))
            if r < 1.5: continue
            stem = Path(lf).stem
            imgs = glob.glob(f"{DATA}/train/images/{stem}.*")
            if not imgs: continue
            ext = Path(imgs[0]).suffix
            for k in range(int(round(r)) - 1):
                dst = f"{DATA}/train/images/{stem}_rfs{k}{ext}"
                if not os.path.exists(dst):
                    shutil.copy(imgs[0], dst)
                    shutil.copy(f"{DATA}/train/labels/{lf}", f"{DATA}/train/labels/{stem}_rfs{k}.txt")
                    added += 1
        print(f"  RFS: +{added} rare-class duplicates")

    names_yaml = "\n".join(f"  {i}: {nm}" for i, nm in enumerate(CLASSES))
    with open(f"{DATA}/data.yaml", "w") as f:
        f.write(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names_yaml}\n")
    print(f"{dst_name}: train={len(glob.glob(f'{DATA}/train/images/*'))} (use_rfs={use_rfs})")
    return f"{DATA}/data.yaml", CLASSES

## Train 2 arm: RFS ON / OFF

In [ ]:
from ultralytics import YOLO
import pandas as pd
RARE = ["no_vest", "boots", "no_boots", "mask", "no_mask"]
rows = []
for arm, use_rfs in [("rfs_on", True), ("rfs_off", False)]:
    key = f"e5_{arm}"
    if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
    dyaml, CLASSES = prepare_dataset(f"ds_{arm}", use_rfs=use_rfs)
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    m.train(data=dyaml, epochs=15, imgsz=640, batch=16, seed=SEED, lr0=0.01,
            project=f"{WORK}/runs", name=f"e5_{arm}", exist_ok=True, plots=False, verbose=False)
    t = m.val(data=dyaml, split="test", verbose=False)
    per = {}
    for idx, ap in zip(t.box.ap_class_index, t.box.ap50):
        per[CLASSES[int(idx)]] = round(float(ap), 4)
    row = {"arm": arm, "mAP50_test": round(t.box.map50, 4), "mAP50_95_test": round(t.box.map, 4),
           **{f"AP50_{c}": per.get(c) for c in RARE}, "min": round((time.time() - t0) / 60, 1)}
    rows.append(row); mark(key, row); print(row)
    del m; free()
    shutil.rmtree(f"{WORK}/ds_{arm}", ignore_errors=True)   # clean up immediately to save disk

df = pd.DataFrame(rows); display(df)
df.to_csv(f"{WORK}/results/e5_ablation_rfs.csv", index=False)
print("Reading the result: RFS should help most in the rare-class AP50_ columns; overall mAP may move only slightly.")

In [ ]:
# ==================== CLEAN UP OUTPUT: no dataset bundled ====================
# Keep: results/, state.json, runs/ (needed to resume the seeds). Delete everything else
# (the copied dataset, the downloaded yolo26s.pt, caches, ...).
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

def du(path):
    t = 0
    for r, _, fs in os.walk(path):
        t += sum(os.path.getsize(os.path.join(r, f)) for f in fs)
    return t
print("Final output:", sorted(os.listdir(WORK)), f"| total {du(WORK)/1e6:.1f} MB (no dataset included)")